# 07: Ligação de pacientes pela agenda (resultado negativo)

Depois das ligações exata (78% dos registros) e aproximada (89%), 11% dos agendamentos seguem sem ID e, portanto,
sem histórico. Sem acesso à fonte da base, a última tentativa é usar o **rastro da agenda**: na reabilitação, as
terapias são semanais e o paciente costuma manter o horário.

**Hipótese:** um registro sem ID pode ser ligado ao paciente que ocupa o mesmo lugar na agenda (dia da semana,
horário, especialidade, gênero e acompanhante) nas semanas vizinhas.

**Como testar sem a resposta:** esconder o ID de pacientes conhecidos numa janela de 3 meses, aplicar a ligação e
medir **precisão** (religados ao paciente certo) e **cobertura** (quantos foram religados). Um ID errado é pior que
nenhum, porque mistura o histórico de duas pessoas, então a precisão é o critério principal.

Código: `src/noshow/linkage.py`. Requisitos: RF-070 a RF-072 (`REQUISITOS.md`).

In [ ]:
import warnings

import numpy as np
import pandas as pd

from noshow.data import load_raw
from noshow.linkage import link_by_schedule, mask_windows, schedule_key, validate_linkage
from noshow.registry import load_bases

warnings.filterwarnings("ignore")
raw = load_raw()
ids = load_bases()["aproximado"].patient_id.astype("Float64").reset_index(drop=True)
agenda = raw[["appointment_date", "appointment_time", "gender", "specialty", "patient_needs_companion"]]
print(f"{len(raw):,} agendamentos | sem ID após as ligações exata e aproximada: {ids.isna().sum():,} ({ids.isna().mean():.1%})")

## 1. O rastro existe, mas não é exclusivo

In [ ]:
k = agenda[ids.notna()].assign(id=ids[ids.notna()]).dropna(subset=["specialty"]).sort_values("appointment_date")
g = k.groupby(["id", "specialty"])
k["proxima"], k["hora_proxima"] = g.appointment_date.shift(-1), g.appointment_time.shift(-1)
seguidas = k.dropna(subset=["proxima"])
semana = seguidas[(seguidas.proxima - seguidas.appointment_date).dt.days == 7]
print(f"Próxima consulta da mesma especialidade 7 dias depois: {len(semana) / len(seguidas):.0%} dos casos")
print(f"  e no mesmo horário: {(semana.appointment_time == semana.hora_proxima).mean():.0%}")

lugar = agenda.appointment_date.astype(str) + "|" + schedule_key(agenda)
compartilhado = lugar.map(lugar.value_counts()) > 1
print(f"\nAgendamentos que dividem data + horário + especialidade + gênero + acompanhante com outro: {compartilhado.mean():.0%}")
print(f"  entre os sem ID: {compartilhado[ids.isna()].mean():.0%} | sem especialidade entre os sem ID: "
      f"{agenda.specialty[ids.isna()].isna().mean():.0%}")

## 2. Validação: precisão × cobertura de cada regra

Em cada regra, 30% dos pacientes conhecidos têm o ID escondido numa janela aleatória de 90 dias. Regras, da mais
permissiva à mais rígida:

- **votos:** em quantas semanas vizinhas (±7 e ±14 dias) o mesmo paciente aparece naquele lugar da agenda;
- **sem conflito:** descarta o candidato que já tem consulta no mesmo dia (portanto não é a pessoa sem ID);
- **propagação:** ligações novas servem de base para a semana seguinte.

O último cenário repete a melhor regra escondendo também a especialidade, como ocorre em 44% dos registros sem ID.

In [ ]:
mascara = mask_windows(ids, raw.appointment_date, frac=0.3, dias=90, seed=0)
REGRAS = {
    "±7 dias, 1 voto": dict(deslocamentos=(7, -7), min_votos=1, excluir_conflito=False, max_passos=1),
    "±7 e ±14 dias, 2 votos": dict(deslocamentos=(7, -7, 14, -14), min_votos=2, excluir_conflito=False, max_passos=1),
    "+ sem conflito": dict(deslocamentos=(7, -7, 14, -14), min_votos=2, excluir_conflito=True, max_passos=1),
    "+ propagação": dict(deslocamentos=(7, -7, 14, -14), min_votos=2, excluir_conflito=True, max_passos=20),
}
linhas = [{"regra": n, **validate_linkage(agenda, ids, mascara, **r)} for n, r in REGRAS.items()]
sem_esp = agenda.assign(specialty=agenda.specialty.mask(mascara))
linhas.append({"regra": "+ propagação, sem especialidade", **validate_linkage(sem_esp, ids, mascara, **REGRAS["+ propagação"])})
validacao = pd.DataFrame(linhas).set_index("regra")
validacao.style.format({"escondidos": "{:,}", "religados": "{:,}", "cobertura": "{:.0%}", "precisao": "{:.1%}"})

## 3. O que a melhor regra faria nos registros reais sem ID

In [ ]:
religado = link_by_schedule(agenda, ids, **REGRAS["+ propagação"])
novos = ids.isna() & religado.notna()
ano = raw.appointment_date.dt.year
impacto = pd.DataFrame({"sem ID": ids.isna().groupby(ano).sum(), "seriam ligados": novos.groupby(ano).sum()})
impacto["cobertura"] = impacto["seriam ligados"] / impacto["sem ID"]
precisao = validacao.loc["+ propagação", "precisao"]
print(f"Seriam ligados {novos.sum():,} de {ids.isna().sum():,} registros sem ID ({novos.sum() / ids.isna().sum():.0%}); "
      f"com a precisão da validação ({precisao:.0%}), cerca de {round(novos.sum() * (1 - precisao)):,} iriam para a pessoa errada")
impacto.style.format({"sem ID": "{:,}", "seriam ligados": "{:,}", "cobertura": "{:.0%}"})

## 4. Decisão

**Não aplicar a ligação por agenda.**

| Evidência | Valor |
|---|---|
| Consultas semanais no mesmo horário | 90%: o rastro existe |
| Agendamentos que dividem o lugar na agenda com outro paciente | 37% (49% entre os sem ID): o rastro não identifica a pessoa |
| Melhor regra na validação | **83% de precisão, 14% de cobertura** |
| Sem especialidade (44% dos registros sem ID) | 26% de precisão, 2% de cobertura |
| Aplicada aos registros reais sem ID | **90 de 5.565 (2%)**, cerca de 15 para a pessoa errada; nenhum em 2022 |

Vários terapeutas atendem no mesmo horário, então o lugar na agenda aponta para mais de um paciente. As regras que
tornam a ligação precisa (exigir duas semanas concordantes e descartar quem já está no dia) eliminam quase todos os
casos. O ganho máximo seria de 2% dos registros sem ID, com 1 erro a cada 6 ligações, o que contaminaria o histórico,
o sinal mais forte do modelo.

**Para o TCC:** a reconstrução do ID foi levada ao limite com o que a base oferece (exato → aproximado → agenda),
cada etapa com a precisão medida. Os 11% restantes ficam declarados como limitação. Em produção, o identificador do
paciente existe no sistema de agendamento, e essa limitação desaparece.